In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ie-506-2025-programming-challenge/data_train.csv
/kaggle/input/ie-506-2025-programming-challenge/sample_submission.csv
/kaggle/input/ie-506-2025-programming-challenge/data_test.csv


## Approach Summary

In this challenge, each sample in the training data had two labels, but only one of them is correct and we don’t know which one. So to handle this, I used both `label1` and `label2` by duplicating the dataset. This way, the model gets to see both versions and can learn better even with some noise.

I built a Multi-Layer Perceptron (MLP) using PyTorch with three hidden layers. I used ReLU activation and dropout (set to 0.25) after each layer to prevent overfitting. The model has 512, 512, and 256 neurons in the hidden layers.

To make the model more accurate, I:
- Normalized the input features
- Used class weights in the loss function to handle imbalance
- Added label smoothing to reduce the effect of wrong labels
- Used weight decay and cosine annealing to help the model generalize better

I trained the model for 50 epochs and after each epoch, I calculated the Macro F1 score on a validation set. I saved the model when it gave the best F1 score, and used that saved version to make predictions on the test set.

Finally, I saved the predictions in the format required for submission.



In [ ]:
#  Step 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

#  Step 2: Load Dataset
train = pd.read_csv('/kaggle/input/ie-506-2025-programming-challenge/data_train.csv')
test = pd.read_csv('/kaggle/input/ie-506-2025-programming-challenge/data_test.csv')

# 🔍 Step 3: Preprocess Features and Labels
feature_cols = [f'x{i}' for i in range(150)]
X1 = train[feature_cols].values
y1 = train['label1'].values
X2 = train[feature_cols].values
y2 = train['label2'].values

X = np.concatenate([X1, X2], axis=0)
y = np.concatenate([y1, y2], axis=0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(test[feature_cols].values)

#  Step 4: Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

#  Step 5: Define Enhanced MLP Model
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(150, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.model(x)

#  Step 6: Train MLP
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP().to(device)

# Compute class weights for balanced loss
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

best_f1 = 0.0
for epoch in range(50):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_tensor.to(device)).argmax(1).cpu().numpy()
        acc = accuracy_score(y_val, val_preds)
        f1 = f1_score(y_val, val_preds, average='macro')
        print(f"Epoch {epoch+1}, Val Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")
        scheduler.step()

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), 'best_model.pth')

#  Step 7: Load Best Model
model.load_state_dict(torch.load('best_model.pth'))

# 🧪 Step 8: Predict on Test Set
model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor.to(device)).argmax(1).cpu().numpy()

#  Step 9: Create Submission File
submission = pd.DataFrame({
    'ID': test['ID'],
    'Class': test_preds
})
submission.to_csv('submission.csv', index=False)



Epoch 1, Val Accuracy: 0.3842, Macro F1: 0.3822
Epoch 2, Val Accuracy: 0.4234, Macro F1: 0.4208
Epoch 3, Val Accuracy: 0.4358, Macro F1: 0.4324
Epoch 4, Val Accuracy: 0.4400, Macro F1: 0.4376
Epoch 5, Val Accuracy: 0.4430, Macro F1: 0.4395
Epoch 6, Val Accuracy: 0.4441, Macro F1: 0.4413
Epoch 7, Val Accuracy: 0.4446, Macro F1: 0.4429
Epoch 8, Val Accuracy: 0.4444, Macro F1: 0.4427
Epoch 9, Val Accuracy: 0.4447, Macro F1: 0.4430
Epoch 10, Val Accuracy: 0.4445, Macro F1: 0.4428
Epoch 11, Val Accuracy: 0.4461, Macro F1: 0.4445
Epoch 12, Val Accuracy: 0.4463, Macro F1: 0.4447
Epoch 13, Val Accuracy: 0.4459, Macro F1: 0.4444
Epoch 14, Val Accuracy: 0.4456, Macro F1: 0.4444
Epoch 15, Val Accuracy: 0.4461, Macro F1: 0.4442
Epoch 16, Val Accuracy: 0.4461, Macro F1: 0.4447
Epoch 17, Val Accuracy: 0.4461, Macro F1: 0.4450
Epoch 18, Val Accuracy: 0.4462, Macro F1: 0.4452
Epoch 19, Val Accuracy: 0.4451, Macro F1: 0.4440
Epoch 20, Val Accuracy: 0.4434, Macro F1: 0.4425
Epoch 21, Val Accuracy: 0.444

/tmp/ipykernel_19/3430750811.py:100: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))
